# Anggota B — Tahap 3: Seleksi Fitur Utama dengan Simulated Annealing (Split Utama / `P0`)

Notebook ini menjalankan **Simulated Annealing (SA)** pada versi normalisasi terpilih dari Tahap 2 (`peak`) menggunakan split utama (`P0`: train aktor 1–16, validation aktor 17–20).

- Fungsi objektif di dalam SA dievaluasi dengan `GroupKFold(n_splits=4)` pada train: `J(s) = CV_macro_F1(s) - lambda * (|s| / 90)`.
- Data `validation` (aktor 17–20) dipakai untuk memverifikasi generalisasi subset terpilih.
- Data `test` (aktor 21–24) tidak disentuh.
- Ukuran subset (`k`) yang dihasilkan dikirim ke Anggota A untuk Langkah 10 (pembanding Mutual Information).


In [ ]:
import json, os, sys
import pandas as pd
import numpy as np
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/project07'
    OUT_DIR = BASE_DIR
except ImportError:
    BASE_DIR = r'E:\College\UB\S2\AI\Projek akhir\AI-Projek'
    OUT_DIR = r'E:\College\UB\S2\AI\Projek akhir\AI-Projek'

sys.path.insert(0, os.path.join(BASE_DIR, 'src'))
sys.path.insert(0, os.path.join(OUT_DIR, 'src'))

import data_splits as ds
import sa_feature_selection as sa


In [ ]:
with open(os.path.join(OUT_DIR, 'results', 'baseline_results.json'), encoding='utf-8') as f:
    bl_data = json.load(f)

norm_ver = bl_data['selected_normalization']
svm_params = bl_data['results'][norm_ver]['SVM']['best_params']
print('Normalisasi:', norm_ver, '| SVM params:', svm_params)

feat_df = pd.read_csv(os.path.join(BASE_DIR, 'data', 'features', f'features_{norm_ver}.csv'))
fnames_df = pd.read_csv(os.path.join(BASE_DIR, 'data', 'features', 'feature_names.csv'))
feat_names = fnames_df['name'].tolist()

tr, va, _ = ds.split_frames(feat_df, ds.load_splits(BASE_DIR))
X_tr, y_tr, g_tr = tr[feat_names].values, tr['emotion'].values, tr['actor'].values
X_va, y_va = va[feat_names].values, va['emotion'].values


In [ ]:
sa_svm_params = {'kernel': 'rbf', 'C': 1.0, 'gamma': 'scale'}
sa_res = sa.run_simulated_annealing(
    X_tr, y_tr, g_tr,
    svm_params=sa_svm_params,
    seed=42,
    n_iter=500,
    t0=0.015,
    t_min=0.0002,
    lambda_penalty=0.015,
    min_features=5,
    stagnation_limit=80,
    max_reheats=2,
    verbose=True,
)
sel_idx = sa_res['best_indices']
print(f'\nJumlah fitur terpilih (k): {len(sel_idx)} / 90')
print('Fitur terpilih:', [feat_names[i] for i in sel_idx])


In [ ]:
clean_params = {k.replace('clf__', ''): v for k, v in svm_params.items()}
pipe = Pipeline([('scaler', StandardScaler()), ('clf', SVC(**clean_params, class_weight='balanced', random_state=42))])
pipe.fit(X_tr[:, sel_idx], y_tr)
val_f1 = f1_score(y_va, pipe.predict(X_va[:, sel_idx]), average='macro')
print(f'Validation Macro F1 (SA subset, k={len(sel_idx)}): {val_f1:.4f}')
